# Phase 1: EDA + toxic-bert Smoke Tests

Explore the Jigsaw 2018 toxic comment dataset and verify `unitary/toxic-bert` loads and runs.

**Data:** `data/raw/jigsaw/` (`train.csv`, `test.csv`, `test_labels.csv`)

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from data import (
    add_toxic_any,
    comment_length_stats,
    label_cooccurrence,
    label_counts,
    load_test_df,
    load_train_df,
    verify_data_files,
)
from paths import LOCAL_MODEL_DIR, MODEL_ID, SEED, TOXIC_LABELS

np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## 1. Data verification

In [ ]:
verify_data_files()

train_df = load_train_df()
test_df = load_test_df()
train_df = add_toxic_any(train_df)
test_df = add_toxic_any(test_df)

print(f"Train rows: {len(train_df):,}")
print(f"Test rows (labeled): {len(test_df):,}")
print(f"Columns: {list(train_df.columns)}")
train_df.head(3)

## 2. Label distribution (EDA)

In [ ]:
counts = label_counts(train_df)
display(counts)

fig, ax = plt.subplots(figsize=(8, 4))
plot_df = counts[counts["label"] != "toxic_any"]
sns.barplot(data=plot_df, x="label", y="pct", ax=ax, palette="viridis")
ax.set_title("Label prevalence (% of train set)")
ax.set_ylabel("%")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 3. Comment length distribution

In [ ]:
length_stats = comment_length_stats(train_df)
display(length_stats)

chars = train_df["comment_text"].astype(str).str.len()
words = train_df["comment_text"].astype(str).str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(chars, bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("Characters per comment")
axes[0].set_xlabel("chars")
axes[1].hist(words, bins=50, color="coral", edgecolor="white")
axes[1].set_title("Words per comment")
axes[1].set_xlabel("words")
plt.tight_layout()
plt.show()

## 4. Label co-occurrence heatmap

Off-diagonal cells count comments where **both** labels are 1. The diagonal is each label's marginal count (label with itself) — not a cross-label pair — so we mask it with `NaN` for the plot.

In [ ]:
cooc = label_cooccurrence(train_df, mask_diagonal=True)
plt.figure(figsize=(7, 6))
sns.heatmap(cooc, annot=True, fmt=".0f", cmap="Blues", mask=cooc.isna())
plt.title("Label co-occurrence (off-diagonal only; diagonal = NaN)")
plt.tight_layout()
plt.show()

## 5. Example comments

In [ ]:
def show_examples(df, label, toxic=True, n=2):
    subset = df[df[label] == (1 if toxic else 0)]
    for _, row in subset.sample(min(n, len(subset)), random_state=SEED).iterrows():
        print(f"[{label}={int(row[label])}] {row['comment_text'][:200]}")
        print("-" * 60)

print("=== Non-toxic examples ===")
show_examples(train_df, "toxic", toxic=False, n=3)

print("\n=== Toxic examples ===")
show_examples(train_df, "toxic", toxic=True, n=3)

print("\n=== identity_hate examples ===")
show_examples(train_df, "identity_hate", toxic=True, n=2)

### Install model dependencies (run once)

If you get `ModuleNotFoundError: transformers`, run the next cell with this notebook's kernel (use `deep_learn`), then **Kernel → Restart** and run all cells from the top.

In [ ]:
# Uncomment and run once if needed:
# %pip install -r ../requirements-project.txt

## 6. Load toxic-bert

Uses local copy in `models/toxic-bert/` if present, otherwise downloads from Hugging Face.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_path = str(LOCAL_MODEL_DIR) if LOCAL_MODEL_DIR.exists() else MODEL_ID
print(f"Loading from: {model_path}")

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")

## 7. Smoke tests — hand-picked sentences

In [ ]:
import torch.nn.functional as F

def predict_toxic_scores(texts):
    """Return sigmoid probabilities for all 6 labels."""
    enc = tokenizer(list(texts), padding=True, truncation=True, max_length=256, return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.sigmoid(logits).cpu().numpy()
    return pd.DataFrame(probs, columns=TOXIC_LABELS, index=texts)

smoke_texts = [
    "You are an idiot and I hate you.",
    "Have a wonderful day, thank you for your help!",
    "I am Muslim and proud of my heritage.",
    "lol that was funny",
    "I will kill you",
]

smoke_df = predict_toxic_scores(smoke_texts)
smoke_df.round(3)

## 8. Smoke test — 100 random train comments

In [ ]:
sample = train_df.sample(100, random_state=SEED)
sample_probs = predict_toxic_scores(sample["comment_text"].tolist())

print("Model P(toxic) distribution on 100 samples:")
display(sample_probs["toxic"].describe())

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(sample_probs["toxic"], bins=20, color="steelblue", edgecolor="white")
ax.set_title("toxic-bert P(toxic) on 100 random train comments")
ax.set_xlabel("P(toxic)")
plt.tight_layout()
plt.show()

## 9. Sanity check — gold label vs model P(toxic)

In [ ]:
compare = sample[["comment_text", "toxic"]].copy()
compare["model_p_toxic"] = sample_probs["toxic"].values
compare["model_pred_toxic"] = (compare["model_p_toxic"] >= 0.5).astype(int)

agreement = (compare["toxic"] == compare["model_pred_toxic"]).mean()
print(f"Agreement on 100 samples (threshold=0.5): {agreement:.1%}")
print("(Not final eval — no val/test split tuning yet)")
compare.head(10)

## Next steps (Phase 2)

- Create train/val/test split (seed=211), save to `data/processed/split.pt`
- Add encoder baselines: DistilBERT, hateBERT, TF-IDF+SVM
- Add decoder prompting: DistilGPT-2
- Perturbation suite and per-identity FPR (Jigsaw 2019 or synthetic templates)
- Distillation: toxic-bert → DistilBERT